# 04 — Modèle Deep Learning (MLP PyTorch)

**Architecture** : Multi-Layer Perceptron (MLP) avec :  
- Couches Dense → BatchNorm → ReLU → Dropout  
- Optimiseur Adam + ReduceLROnPlateau  
- Perte BCELoss (classification binaire)  

**Données** : mêmes features preprocessées que les modèles ML (26 features)

### 1 - Chargement des données 

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

PROC_PATH = Path('../data/processed')
DL_PATH   = Path('../models/dl')
FIGS_PATH = Path('../reports/figures')
DL_PATH.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

# Chargement
X_train = np.load(PROC_PATH / 'X_train.npy').astype(np.float32)
X_test  = np.load(PROC_PATH / 'X_test.npy').astype(np.float32)
y_train = np.load(PROC_PATH / 'y_train.npy').astype(np.float32)
y_test  = np.load(PROC_PATH / 'y_test.npy').astype(np.float32)
feat_names = np.load(PROC_PATH / 'feature_names.npy', allow_pickle=True)

print(f'X_train : {X_train.shape}  y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}   y_test  : {y_test.shape}')
print(f'Input dim : {X_train.shape[1]}')

Device : cpu
X_train : (814, 26)  y_train : (814,)
X_test  : (184, 26)   y_test  : (184,)
Input dim : 26


### 2 - DataLoaders PyTorch

In [15]:
BATCH_SIZE = 32

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Batches train : {len(train_dl)}  |  Batches test : {len(test_dl)}')

Batches train : 26  |  Batches test : 6


### 3 - Architecture du réseau

In [16]:
class HeartMLP(nn.Module):
    """
    MLP pour classification binaire.
    Architecture : Input → [Linear → BatchNorm → ReLU → Dropout] × N → Linear → Sigmoid
    """
    def __init__(self, input_dim: int, hidden_dims: list, dropout: float = 0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


# Instanciation
INPUT_DIM   = X_train.shape[1]
HIDDEN_DIMS = [128, 64, 32]
DROPOUT     = 0.3

model = HeartMLP(INPUT_DIM, HIDDEN_DIMS, DROPOUT).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal paramètres    : {total_params:,}')
print(f'Paramètres entraînables : {train_params:,}')

HeartMLP(
  (net): Sequential(
    (0): Linear(in_features=26, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
    (13): Sigmoid()
  )
)

Total paramètres    : 14,273
Paramètres entraînables : 14,273


### 4 - Fonctions d'entraînement & d'évaluation

In [17]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        preds = model(xb)
        loss  = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
        correct    += ((preds >= 0.5).float() == yb).sum().item()
        total      += len(xb)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_proba, all_labels = [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        proba = model(xb)
        loss  = criterion(proba, yb)
        total_loss += loss.item() * len(xb)
        correct    += ((proba >= 0.5).float() == yb).sum().item()
        total      += len(xb)
        all_proba.extend(proba.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    return (total_loss / total, correct / total,
            np.array(all_labels), np.array(all_proba))

### 4 - Entraînement

In [18]:
EPOCHS = 120
LR     = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=10, factor=0.5
)
criterion = nn.BCELoss()

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_auc':[]}
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc             = train_epoch(model, train_dl, optimizer, criterion)
    val_loss, val_acc, lbl, prb = eval_epoch(model, test_dl, criterion)
    val_auc                     = roc_auc_score(lbl, prb)
    scheduler.step(val_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), DL_PATH / 'best_mlp.pt')

    if epoch % 20 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  '
              f'val_acc={val_acc:.4f}  val_auc={val_auc:.4f}  '
              f'lr={lr_now:.6f}')

print(f'\nMeilleure val_acc : {best_acc:.4f}  →  modèle sauvegardé')

Epoch   1/120  train_loss=0.6212  val_loss=0.5424  val_acc=0.7935  val_auc=0.8920  lr=0.001000
Epoch  20/120  train_loss=0.3392  val_loss=0.3634  val_acc=0.8478  val_auc=0.9149  lr=0.001000
Epoch  40/120  train_loss=0.2765  val_loss=0.3670  val_acc=0.8424  val_auc=0.9191  lr=0.000250
Epoch  60/120  train_loss=0.2722  val_loss=0.3749  val_acc=0.8370  val_auc=0.9158  lr=0.000063
Epoch  80/120  train_loss=0.2616  val_loss=0.3720  val_acc=0.8370  val_auc=0.9156  lr=0.000031
Epoch 100/120  train_loss=0.2616  val_loss=0.3722  val_acc=0.8315  val_auc=0.9163  lr=0.000008
Epoch 120/120  train_loss=0.2743  val_loss=0.3770  val_acc=0.8424  val_auc=0.9155  lr=0.000002

Meilleure val_acc : 0.8641  →  modèle sauvegardé
